In [5]:
# -*- coding: utf-8 -*-                                   # Codificação do arquivo (acentos PT-BR)
# Script: Extração de voltas (laps) FastF1 2022–2024      # Descrição breve
# Objetivo: Usar cache SOMENTE em notebooks/fastf1_cache  # Padronização do cache

from pathlib import Path                                   # Manipulação de caminhos de forma portátil
import pandas as pd                                        # DataFrames e IO
import numpy as np                                         # Utilidades numéricas
import fastf1                                             # Biblioteca FastF1 (dados de F1)

# =========================
# Parâmetros gerais
# =========================

YEARS = [2022, 2023, 2024]                                 # Anos-alvo para extração
SKIP_IF_SAVED = True                                       # Se True, não reextrai quando já existe arquivo salvo

# =========================
# Estrutura de pastas (sempre usar notebooks/fastf1_cache)
# =========================
CWD = Path.cwd().resolve()                                 # Diretório corrente absoluto
if CWD.name.lower() == "notebooks":                        # Caso o notebook esteja dentro de /notebooks
    NB_DIR = CWD                                           # NB_DIR = .../notebooks
else:                                                      # Caso o notebook rode a partir da raiz do repositório
    NB_DIR = CWD / "notebooks"                             # NB_DIR = .../notebooks (força o uso dessa pasta)

DATA_DIR = NB_DIR / "data"                                 # Saídas e dados em notebooks/data
DATA_DIR.mkdir(parents=True, exist_ok=True)                # Garante que a pasta exista

INTERIM = DATA_DIR / "interim"                             # Subpasta para arquivos intermediários
INTERIM.mkdir(parents=True, exist_ok=True)                 # Garante que a pasta exista

CACHE_DIR = NB_DIR / "fastf1_cache"                        # **ÚNICO** cache suportado: notebooks/fastf1_cache
CACHE_DIR.mkdir(parents=True, exist_ok=True)               # Garante que a pasta exista
fastf1.Cache.enable_cache(str(CACHE_DIR))                  # Ativa cache do FastF1 nesse caminho

print(f"Cache ativo em: {CACHE_DIR}")                      # Loga caminho do cache efetivo
print(f"Saídas em:      {INTERIM}")                        # Loga caminho das saídas

# =========================
# Circuitos-alvo (mapeamento por Location e EventName)
# =========================
TARGETS = {
    # Já existentes
    "Bahrein":     {"location": ["sakhir", "bahrain"],            "event_sub": ["bahrain"]},
    "Jeddah":      {"location": ["jeddah"],                       "event_sub": ["saudi"]},
    "Australia":   {"location": ["melbourne", "australia"],       "event_sub": ["australian"]},
    "Baku":        {"location": ["baku"],                         "event_sub": ["azerbaijan"]},
    "Miami":       {"location": ["miami"],                        "event_sub": ["miami"]},
    "Monza":       {"location": ["monza"],                        "event_sub": ["italian"]},
    "Singapura":   {"location": ["singapore"],                    "event_sub": ["singapore"]},
    "Suzuka":      {"location": ["suzuka"],                       "event_sub": ["japan"]},
    "COTA":        {"location": ["austin"],                       "event_sub": ["united states"]},   # COTA/Austin (EUA)
    "Mexico":      {"location": ["mexico city"],                  "event_sub": ["mexico"]},
    "Brasil":      {"location": ["são paulo", "sao paulo"],       "event_sub": ["sao paulo", "brazil"]},
    "Abu Dhabi":   {"location": ["abu dhabi", "yas marina"],      "event_sub": ["abu dhabi"]},
    "Silverstone": {"location": ["silverstone"],                  "event_sub": ["british", "great britain"]},
    "Bélgica":     {"location": ["spa-francorchamps", "spa"],     "event_sub": ["belgian"]},
    "Hungria":     {"location": ["hungaroring", "budapest"],      "event_sub": ["hungarian"]},
    "Mônaco":      {"location": ["monaco"],                       "event_sub": ["monaco"]},
}

# =========================
# Carrega calendários (sem testes)
# =========================
all_events = []                                             # Buffer para todos os anos

for y in YEARS:                                             # Itera 2022, 2023, 2024
    cal = fastf1.get_event_schedule(y, include_testing=False).copy()  # Busca calendário e copia
    keep_cols = [c for c in ["RoundNumber","EventName","OfficialEventName","EventDate","Location","EventFormat"] if c in cal.columns]
                                                              # Mantém apenas colunas relevantes (as que existirem)
    cal = cal[keep_cols]                                    # Aplica o filtro de colunas
    cal["Year"] = y                                         # Marca o ano
    cal["loc_lc"] = cal["Location"].fillna("").str.lower()  # Normaliza Location (minúsculas) p/ matching
    cal["ename_lc"] = cal["EventName"].fillna("").str.lower()  # Normaliza EventName (minúsculas) p/ matching
    all_events.append(cal)                                  # Acumula esse calendário

events_df = pd.concat(all_events, ignore_index=True)        # Concatena calendários dos 3 anos em um DF

# =========================
# Função: regra de matching (linha do calendário ↔ circuito)
# =========================
def row_matches_target(row, target) -> bool:                # Retorna True se a linha pertence ao circuito-alvo
    loc = row["loc_lc"]                                     # Texto de Location em minúsculas
    enm = row["ename_lc"]                                   # Texto de EventName em minúsculas
    if any(sub in loc for sub in target["location"]):       # Bate por fragmentos na Location
        return True                                         # Se achou, casa
    if any(sub in enm for sub in target["event_sub"]):      # Ou bate por fragmentos no EventName
        return True                                         # Se achou, casa
    return False                                            # Caso contrário, não casa

# =========================
# Função: extrai voltas da corrida (session = 'R')
# =========================
def extract_event_race_laps(year: int, round_number: int | None, event_name: str | None) -> pd.DataFrame:
    """Carrega a sessão 'R' (Race) do evento e retorna o DataFrame de voltas (laps) com metadados."""
    if pd.notna(round_number):                              # Se tiver RoundNumber (mais robusto)
        ses = fastf1.get_session(int(year), int(round_number), 'R')  # Pega pela etapa
    else:                                                   # Senão, tenta pelo nome do evento
        ses = fastf1.get_session(int(year), str(event_name), 'R')    # (menos robusto, mas funciona)

    ses.load()                                              # Carrega dados (usa/gera cache em notebooks/fastf1_cache)
    laps = ses.laps.copy()                                  # Copia o DF de voltas

    laps["Year"] = year                                     # Marca o ano
    laps["RoundNumber"] = int(round_number) if pd.notna(round_number) else np.nan  # Marca a etapa (ou NaN)
    laps["EventName"] = ses.event.EventName if hasattr(ses, "event") else event_name  # Nome oficial do evento
    laps["SessionName"] = "Race"                            # Indica que é corrida (não Sprint/Quali)
    return laps                                             # Retorna DF

# =========================
# Loop principal: para cada circuito, extrair/ler e salvar
# =========================
all_circuits_laps = []                                      # Para montar o dataset mestre (todos circuitos)

for circuito, matchers in TARGETS.items():                  # Itera pelos 16 circuitos
    safe_circuit = circuito.lower().replace(" ", "_")       # Nome “seguro” para arquivo
    out_csv  = INTERIM / f"{safe_circuit}_2022-2024_all_laps.csv"      # Caminho CSV de saída
    out_parq = INTERIM / f"{safe_circuit}_2022-2024_all_laps.parquet"  # Caminho Parquet de saída

    # --- ETAPA 0: pular se já existe (não chama API) ---
    if SKIP_IF_SAVED and (out_parq.exists() or out_csv.exists()):      # Se já existe saída salva
        try:                                                           # Tenta ler
            if out_parq.exists():                                      # Prefere Parquet (rápido/leve)
                laps_circuito = pd.read_parquet(out_parq)              # Lê Parquet
                print(f"[PULADO] {circuito}: carregado de Parquet existente.")  # Log
            else:                                                      # Senão, lê CSV
                laps_circuito = pd.read_csv(out_csv)                   # Lê CSV
                print(f"[PULADO] {circuito}: carregado de CSV existente.")      # Log
            all_circuits_laps.append(laps_circuito)                    # Acumula no mestre
            continue                                                   # Vai ao próximo circuito
        except Exception as e:                                         # Se falhar leitura
            print(f"[AVISO] Falha ao ler saída existente de '{circuito}' ({e}). Reextraindo...")  # Avisa e segue

    # --- ETAPA 1: localizar eventos do circuito no calendário ---
    mask = events_df.apply(lambda r: row_matches_target(r, matchers), axis=1)  # Aplica matching linha a linha
    cal_hits = events_df[mask].copy().sort_values(["Year","RoundNumber","EventDate"])  # Eventos encontrados ordenados

    if cal_hits.empty:                                                # Se não achou nada
        print(f"[AVISO] Nenhum evento encontrado para '{circuito}' nos anos {YEARS}.")  # Loga aviso
        continue                                                      # Próximo circuito

    print(f"\n=== {circuito}: {len(cal_hits)} evento(s) 2022–2024 ===")  # Log informativo
    per_circuit_laps = []                                             # Buffer de voltas desse circuito
    errors = []                                                       # Buffer de erros (se houver)

    # --- ETAPA 2: para cada evento (ano), extrair voltas da corrida ---
    for _, ev in cal_hits.iterrows():                                 # Itera pelos eventos localizados
        y = int(ev["Year"])                                           # Ano do evento
        rnd = ev["RoundNumber"] if "RoundNumber" in ev and pd.notna(ev["RoundNumber"]) else None  # Round (ou None)
        ename = ev["EventName"]                                       # Nome do evento (fallback/log)

        try:                                                          # Tenta extrair laps
            df_laps = extract_event_race_laps(y, rnd, ename)          # Extrai voltas da sessão 'R'
            df_laps["Circuito"] = circuito                            # Marca circuito
            df_laps["EventDate"] = pd.to_datetime(ev["EventDate"]).date() if pd.notna(ev["EventDate"]) else pd.NaT
                                                                       # Data do evento como date
            df_laps["Location"] = ev["Location"]                      # Cidade/pista (Location)
            df_laps["OfficialEventName"] = ev.get("OfficialEventName", np.nan)  # Nome oficial completo
            per_circuit_laps.append(df_laps)                          # Acumula
            print(f"[OK] {circuito} {y} (Round {rnd if rnd is not None else 'n/a'}): {len(df_laps)} voltas.")
        except Exception as e:                                        # Trata erro
            print(f"[ERRO] {circuito} {y} (Round {rnd if rnd is not None else 'n/a'}): {e}")
            errors.append((circuito, y, str(e)))                      # Guarda info do erro

    # --- ETAPA 3: consolidar e salvar saídas do circuito ---
    if per_circuit_laps:                                              # Se coletou algo
        laps_circuito = pd.concat(per_circuit_laps, ignore_index=True)  # Concatena anos do circuito
        sort_cols = [c for c in ["Year","Driver","LapNumber"] if c in laps_circuito.columns]  # Colunas p/ ordenação
        if sort_cols:                                                 # Se existirem
            laps_circuito = laps_circuito.sort_values(sort_cols).reset_index(drop=True)  # Ordena

        laps_circuito.to_csv(out_csv, index=False, encoding="utf-8")  # Salva CSV
        laps_circuito.to_parquet(out_parq, index=False)               # Salva Parquet
        print(f"[SALVO] {circuito}:")                                  # Log de salvamento
        print(f"        CSV:     {out_csv}")                           # Caminho CSV
        print(f"        Parquet: {out_parq}")                          # Caminho Parquet

        all_circuits_laps.append(laps_circuito)                        # Acumula no mestre
    else:                                                              # Se nada foi extraído
        print(f"[AVISO] Nenhuma volta consolidada para '{circuito}'. Erros: {len(errors)}")  # Loga aviso

# =========================
# Dataset mestre (todos circuitos juntos)
# =========================
if all_circuits_laps:                                                 # Se houve algum sucesso
    master = pd.concat(all_circuits_laps, ignore_index=True)          # Concatena tudo

    preview_cols = [c for c in ["Circuito","Year","EventName","Driver","LapNumber","LapTime","Compound","Stint","PitInTime","PitOutTime"] if c in master.columns]
                                                                       # Colunas sugeridas para preview
    print("\n=== AMOSTRA (dataset mestre) ===")                       # Título do preview
    if preview_cols:                                                  # Se colunas existirem
        print(master[preview_cols].head(12).to_string(index=False))   # Mostra 12 linhas
    else:                                                             # Caso contrário
        print(master.head(12).to_string(index=False))                 # Mostra 12 linhas genéricas

    master_csv  = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.csv"      # Caminho CSV mestre
    master_parq = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.parquet"  # Caminho Parquet mestre
    master.to_csv(master_csv, index=False, encoding="utf-8")          # Salva CSV mestre
    master.to_parquet(master_parq, index=False)                       # Salva Parquet mestre

    print(f"\n[SALVO] Dataset mestre:")                               # Confirmação
    print(f"        CSV:     {master_csv}")                           # Caminho CSV
    print(f"        Parquet: {master_parq}")                          # Caminho Parquet
else:                                                                  # Se nenhum circuito produziu dados
    raise RuntimeError("Nenhuma volta foi extraída para os circuitos solicitados.")  # Erro explícito


Cache ativo em: C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\fastf1_cache
Saídas em:      C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\data\interim
[PULADO] Bahrein: carregado de Parquet existente.
[PULADO] Jeddah: carregado de Parquet existente.
[PULADO] Australia: carregado de Parquet existente.
[PULADO] Baku: carregado de Parquet existente.
[PULADO] Miami: carregado de Parquet existente.
[PULADO] Monza: carregado de Parquet existente.
[PULADO] Singapura: carregado de Parquet existente.
[PULADO] Suzuka: carregado de Parquet existente.
[PULADO] COTA: carregado de Parquet existente.
[PULADO] Mexico: carregado de Parquet existente.
[PULADO] Brasil: carregado de Parquet existente.
[PULADO] Abu Dhabi: carregado de Parquet existente.
[PULADO] Silverstone: carregado de Parquet existente.
[PULADO] Bélgica: carregado de Parquet existente.
[PULADO] Hungria: carregad